In [ ]:
# ================================================================
# QWEN-IMAGE-2.1 — COMPLETE ONE-CELL KAGGLE T4 SETUP
# ================================================================
#
# ONE CELL DOES EVERYTHING:
#
# 1. Stops an old app.py process
# 2. Removes incompatible torchao
# 3. Cleans old AI/UI packages
# 4. Does NOT remove Kaggle's PyTorch/CUDA
# 5. Installs required packages
# 6. Creates app.py automatically
# 7. Loads Qwen-Image-2.1
# 8. Uses 4-bit NF4 + FP16 for T4
# 9. Creates persistent Gradio UI
# 10. Creates generated image files
# 11. Shows a public Gradio URL
#
# IMPORTANT:
# Run this cell ONCE after starting a fresh Kaggle session.
#
# ================================================================


import os
import sys
import subprocess
import textwrap
import time
import shutil

# ================================================================
# CONFIGURATION
# ================================================================

APP_FILE = "/kaggle/working/app.py"

OUTPUT_DIR = "/kaggle/working/generated_images"

MODEL_ID = "Qwen/Qwen-Image-2.1"


# ================================================================
# 1. STOP PREVIOUS APP PROCESS
# ================================================================

print("=" * 70)
print("STEP 1/6 — STOPPING OLD APP PROCESS")
print("=" * 70)

try:

    result = subprocess.run(
        ["ps", "-eo", "pid,args"],
        text=True,
        capture_output=True
    )

    killed = 0

    for line in result.stdout.splitlines():

        line = line.strip()

        if not line:
            continue

        parts = line.split(None, 1)

        if len(parts) != 2:
            continue

        pid_text, command = parts

        try:
            pid = int(pid_text)
        except ValueError:
            continue

        # Only target our generated app.py.
        # Do NOT touch the notebook kernel.

        if (
            "python" in command
            and "/kaggle/working/app.py" in command
            and pid != os.getpid()
        ):

            print(
                f"🛑 Stopping old app process: {pid}"
            )

            try:

                os.kill(
                    pid,
                    15
                )

                killed += 1

            except Exception:
                pass

    if killed == 0:
        print(
            "✅ No previous app.py process found."
        )
    else:
        print(
            f"✅ Stopped {killed} old process(es)."
        )

except Exception as e:

    print(
        "⚠️ Could not inspect old processes:",
        e
    )


# ================================================================
# 2. REMOVE CONFLICTING PACKAGES
# ================================================================

print()
print("=" * 70)
print("STEP 2/6 — CLEANING OLD / CONFLICTING PACKAGES")
print("=" * 70)


# ------------------------------------------------
# IMPORTANT:
#
# We DO NOT uninstall:
#
#   torch
#   torchvision
#   torchaudio
#
# because Kaggle's CUDA installation is our base.
#
# ------------------------------------------------

REMOVE_PACKAGES = [
    "torchao",
    "diffusers",
    "transformers",
    "bitsandbytes",
    "accelerate",
    "safetensors",
    "huggingface_hub",
    "gradio",
    "Pillow",
]


for package in REMOVE_PACKAGES:

    print(
        f"🧹 Removing {package}..."
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "uninstall",
            "-y",
            package,
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    )


print(
    "✅ Conflicting packages removed."
)


# ================================================================
# 3. INSTALL CLEAN DEPENDENCIES
# ================================================================

print()
print("=" * 70)
print("STEP 3/6 — INSTALLING CLEAN DEPENDENCIES")
print("=" * 70)


INSTALL_PACKAGES = [

    # Qwen Image 2.1 needs recent Transformers
    "transformers>=5.17.0",

    # Device / memory management
    "accelerate",

    # 4-bit quantization
    "bitsandbytes",

    # Model files
    "safetensors",

    # HuggingFace downloading
    "huggingface_hub",

    # Avoid the previous PIL _Ink mismatch
    "pillow==11.3.0",

    # Web UI
    "gradio",
]


for package in INSTALL_PACKAGES:

    print(
        f"📦 Installing {package}..."
    )

    result = subprocess.run(

        [
            sys.executable,
            "-m",
            "pip",
            "install",

            # Quiet output
            "-q",

            # Always get the requested version
            "-U",

            # Avoid stale cache
            "--no-cache-dir",

            package,
        ],

        text=True,

        capture_output=True,
    )


    if result.returncode != 0:

        print()
        print(result.stdout)
        print(result.stderr)

        raise RuntimeError(
            f"❌ Failed to install {package}"
        )


# ------------------------------------------------
# Install latest Diffusers
#
# --no-deps is intentional:
#
# We already installed the dependencies above,
# and we do NOT want pip to change Kaggle's
# PyTorch/CUDA stack.
#
# ------------------------------------------------

print()
print(
    "📦 Installing current Diffusers..."
)


result = subprocess.run(

    [
        sys.executable,
        "-m",
        "pip",
        "install",

        "-q",
        "-U",
        "--no-cache-dir",

        "--no-deps",

        "git+https://github.com/huggingface/diffusers.git",
    ],

    text=True,

    capture_output=True,
)


if result.returncode != 0:

    print(result.stdout)
    print(result.stderr)

    raise RuntimeError(
        "❌ Diffusers installation failed."
    )


print()
print(
    "✅ Clean dependencies installed."
)


# ================================================================
# 4. PREPARE OUTPUT DIRECTORY
# ================================================================

print()
print("=" * 70)
print("STEP 4/6 — PREPARING FILE STRUCTURE")
print("=" * 70)


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


print(
    "✅ Output directory:",
    OUTPUT_DIR
)


# ================================================================
# 5. CREATE app.py
# ================================================================

print()
print("=" * 70)
print("STEP 5/6 — CREATING app.py")
print("=" * 70)


app_code = r'''
# ================================================================
# QWEN-IMAGE-2.1 — PERSISTENT GRADIO SERVER
# ================================================================

import os
import gc
import time
import threading
from datetime import datetime

# ------------------------------------------------------------
# Environment settings
# ------------------------------------------------------------

os.environ[
    "PYTORCH_CUDA_ALLOC_CONF"
] = "expandable_segments:True"

os.environ[
    "TOKENIZERS_PARALLELISM"
] = "false"


# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------

import torch
import gradio as gr

from diffusers import QwenImage21Pipeline

from diffusers.quantizers import (
    PipelineQuantizationConfig
)


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_ID = "Qwen/Qwen-Image-2.1"

OUTPUT_DIR = "/kaggle/working/generated_images"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# GPU CHECK
# ============================================================

print()
print("=" * 70)
print("QWEN-IMAGE-2.1 — GPU CHECK")
print("=" * 70)


if not torch.cuda.is_available():

    raise RuntimeError(
        "❌ CUDA GPU was not detected.\n\n"
        "Please select a Kaggle GPU runtime."
    )


GPU_NAME = torch.cuda.get_device_name(0)


GPU_MEMORY = (
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3)
)


print(
    "GPU :",
    GPU_NAME
)

print(
    "VRAM:",
    round(GPU_MEMORY, 2),
    "GB"
)

print(
    "PyTorch:",
    torch.__version__
)

print(
    "CUDA:",
    torch.version.cuda
)

print(
    "BF16 supported:",
    torch.cuda.is_bf16_supported()
)


# ============================================================
# 4-BIT NF4 CONFIGURATION
# ============================================================
#
# Transformer:
#     4-bit NF4
#
# Text encoder:
#     4-bit NF4
#
# VAE:
#     FP16
#
# Compute:
#     FP16
#
# T4:
#     16 GB class GPU
#
# ============================================================

print()
print("=" * 70)
print("CREATING 4-BIT NF4 CONFIGURATION")
print("=" * 70)


quant_config = PipelineQuantizationConfig(

    quant_backend="bitsandbytes_4bit",

    quant_kwargs={

        "load_in_4bit": True,

        "bnb_4bit_quant_type": "nf4",

        # ----------------------------------------------------
        # IMPORTANT FOR T4
        # ----------------------------------------------------
        # Use FP16 compute instead of BF16.
        # ----------------------------------------------------

        "bnb_4bit_compute_dtype": torch.float16,

        "bnb_4bit_use_double_quant": True,
    },

    components_to_quantize=[
        "transformer",
        "text_encoder",
    ],
)


print(
    "✅ Transformer: 4-bit NF4"
)

print(
    "✅ Text encoder: 4-bit NF4"
)

print(
    "✅ Compute: FP16"
)

print(
    "✅ TorchAO: NOT USED"
)


# ============================================================
# LOAD MODEL
# ============================================================

print()
print("=" * 70)
print("LOADING QWEN-IMAGE-2.1")
print("=" * 70)


print(
    "Model:",
    MODEL_ID
)

print(
    "This happens ONCE."
)

print(
    "After loading, the UI can generate multiple images."
)

print()


load_start = time.time()


# ------------------------------------------------------------
# Load Qwen Image 2.1
# ------------------------------------------------------------

pipe = QwenImage21Pipeline.from_pretrained(

    MODEL_ID,

    # --------------------------------------------------------
    # T4-friendly dtype
    # --------------------------------------------------------

    dtype=torch.float16,

    # --------------------------------------------------------
    # BitsAndBytes NF4
    # --------------------------------------------------------

    quantization_config=quant_config,

    # --------------------------------------------------------
    # Reduce CPU memory spikes
    # --------------------------------------------------------

    low_cpu_mem_usage=True,

    # --------------------------------------------------------
    # GPU
    # --------------------------------------------------------

    device_map="cuda",
)


# ============================================================
# VAE MEMORY OPTIMIZATION
# ============================================================

if (
    hasattr(pipe, "vae")
    and pipe.vae is not None
):

    try:

        pipe.vae.enable_tiling()

        print(
            "✅ VAE tiling enabled."
        )

    except Exception as e:

        print(
            "⚠️ VAE tiling unavailable:",
            e
        )


    try:

        pipe.vae.enable_slicing()

        print(
            "✅ VAE slicing enabled."
        )

    except Exception as e:

        print(
            "⚠️ VAE slicing unavailable:",
            e
        )


# ============================================================
# MODEL MEMORY REPORT
# ============================================================

load_time = (
    time.time() - load_start
)


allocated = (
    torch.cuda.memory_allocated()
    / (1024 ** 3)
)


reserved = (
    torch.cuda.memory_reserved()
    / (1024 ** 3)
)


print()
print("=" * 70)
print("✅ MODEL LOADED SUCCESSFULLY")
print("=" * 70)


print(
    "Load time:",
    round(load_time / 60, 2),
    "minutes"
)


print(
    "GPU allocated:",
    round(allocated, 2),
    "GB"
)


print(
    "GPU reserved:",
    round(reserved, 2),
    "GB"
)


# ============================================================
# GENERATION LOCK
# ============================================================
#
# Only allow one generation at a time.
#
# This prevents accidental double-clicks from
# overloading the T4.
#
# ============================================================

generation_lock = threading.Lock()


# ============================================================
# IMAGE GENERATION FUNCTION
# ============================================================

def generate_image(
    prompt,
    steps,
    width,
    height,
    seed,
):

    # --------------------------------------------------------
    # Prompt validation
    # --------------------------------------------------------

    if prompt is None:

        return (
            None,
            "❌ Prompt is empty."
        )


    prompt = prompt.strip()


    if not prompt:

        return (
            None,
            "❌ Please enter a prompt."
        )


    # --------------------------------------------------------
    # One generation at a time
    # --------------------------------------------------------

    if not generation_lock.acquire(
        blocking=False
    ):

        return (
            None,
            "⏳ Another image is currently generating.\n"
            "Please wait for it to finish."
        )


    try:

        # ----------------------------------------------------
        # Cleanup temporary GPU allocations
        # ----------------------------------------------------

        gc.collect()

        torch.cuda.empty_cache()


        # ----------------------------------------------------
        # Convert UI values
        # ----------------------------------------------------

        steps = int(steps)

        width = int(width)

        height = int(height)

        seed = int(seed)


        # ----------------------------------------------------
        # Resolution protection
        #
        # T4 has 14.56-ish usable GB in this environment.
        #
        # Start with:
        #
        # 768 x 1024
        #
        # ----------------------------------------------------

        width = max(
            512,
            min(width, 1024)
        )

        height = max(
            512,
            min(height, 1024)
        )


        # ----------------------------------------------------
        # Step protection
        # ----------------------------------------------------

        steps = max(
            10,
            min(steps, 40)
        )


        # ----------------------------------------------------
        # Seed
        # ----------------------------------------------------

        generator = torch.Generator(
            device="cuda"
        ).manual_seed(seed)


        print()
        print("=" * 70)
        print("🎨 STARTING IMAGE GENERATION")
        print("=" * 70)

        print(
            "Prompt:",
            prompt
        )

        print(
            "Resolution:",
            f"{width} × {height}"
        )

        print(
            "Steps:",
            steps
        )

        print(
            "Seed:",
            seed
        )


        # ----------------------------------------------------
        # Generate
        # ----------------------------------------------------

        start = time.time()


        with torch.inference_mode():

            result = pipe(

                prompt=prompt,

                width=width,

                height=height,

                num_inference_steps=steps,

                # ------------------------------------------------
                # Keep at 1.0 for this T4 configuration
                # ------------------------------------------------

                true_cfg_scale=1.0,

                generator=generator,
            )


        # ----------------------------------------------------
        # Get PIL image
        # ----------------------------------------------------

        image = result.images[0]


        # ----------------------------------------------------
        # Create unique filename
        # ----------------------------------------------------

        timestamp = datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )

        filename = (
            f"qwen_{timestamp}_"
            f"{seed}.png"
        )


        output_path = os.path.join(
            OUTPUT_DIR,
            filename
        )


        # ----------------------------------------------------
        # Save file
        # ----------------------------------------------------

        image.save(
            output_path
        )


        elapsed = (
            time.time() - start
        )


        # ----------------------------------------------------
        # Memory report
        # ----------------------------------------------------

        current_memory = (
            torch.cuda.memory_allocated()
            / (1024 ** 3)
        )


        status = (
            "✅ Image generated successfully\n\n"
            f"File: {filename}\n"
            f"Resolution: {width} × {height}\n"
            f"Steps: {steps}\n"
            f"Seed: {seed}\n"
            f"Generation time: {elapsed:.1f} seconds\n"
            f"GPU memory: {current_memory:.2f} GB"
        )


        print()
        print(status)


        return (
            image,
            status
        )


    # ========================================================
    # GPU OUT OF MEMORY
    # ========================================================

    except torch.cuda.OutOfMemoryError:

        print(
            "❌ CUDA OUT OF MEMORY"
        )


        torch.cuda.empty_cache()

        gc.collect()


        return (
            None,

            "❌ GPU OUT OF MEMORY.\n\n"
            "Try:\n"
            "• Width = 512\n"
            "• Height = 512\n"
            "• Steps = 15–20\n"
            "• Only one generation at a time"
        )


    # ========================================================
    # OTHER ERROR
    # ========================================================

    except Exception as e:

        print()
        print("=" * 70)
        print("❌ GENERATION ERROR")
        print("=" * 70)

        print(
            repr(e)
        )


        return (
            None,

            "❌ Generation failed:\n\n"
            + str(e)
        )


    finally:

        # ----------------------------------------------------
        # Clean temporary memory.
        #
        # DO NOT delete pipe.
        #
        # The model stays loaded.
        # ----------------------------------------------------

        gc.collect()

        torch.cuda.empty_cache()

        generation_lock.release()


# ============================================================
# GRADIO USER INTERFACE
# ============================================================

print()
print("=" * 70)
print("BUILDING WEB UI")
print("=" * 70)


with gr.Blocks(
    title="Qwen Image 2.1 Generator"
) as demo:


    # ========================================================
    # HEADER
    # ========================================================

    gr.Markdown(
        """
        # 🎨 Qwen-Image-2.1

        ### Photorealistic AI Image Generator

        Enter a prompt and click **Generate Image**.

        **The model stays loaded in the T4 GPU, so you do not
        need to reload the model for every image.**
        """
    )


    # ========================================================
    # MAIN AREA
    # ========================================================

    with gr.Row():


        # ====================================================
        # LEFT COLUMN
        # ====================================================

        with gr.Column(
            scale=1
        ):


            prompt = gr.Textbox(

                label="Image Prompt",

                placeholder=(
                    "Describe exactly what you want..."
                ),

                lines=8,

                value=(
                    "A highly photorealistic cinematic "
                    "portrait of a young Indian man "
                    "standing on a modern Mumbai rooftop "
                    "during blue hour, realistic natural "
                    "skin texture, individual hair strands, "
                    "realistic facial proportions, detailed "
                    "clothing fabric, soft cinematic lighting, "
                    "shallow depth of field, realistic city "
                    "background, professional full-frame "
                    "camera photography, natural colors"
                ),
            )


            # ------------------------------------------------
            # WIDTH / HEIGHT
            # ------------------------------------------------

            with gr.Row():

                width = gr.Slider(

                    minimum=512,

                    maximum=1024,

                    value=768,

                    step=64,

                    label="Width",
                )


                height = gr.Slider(

                    minimum=512,

                    maximum=1024,

                    value=1024,

                    step=64,

                    label="Height",
                )


            # ------------------------------------------------
            # STEPS / SEED
            # ------------------------------------------------

            with gr.Row():

                steps = gr.Slider(

                    minimum=10,

                    maximum=40,

                    value=25,

                    step=1,

                    label="Inference Steps",
                )


                seed = gr.Number(

                    value=42,

                    precision=0,

                    label="Seed",
                )


            # ------------------------------------------------
            # GENERATE BUTTON
            # ------------------------------------------------

            generate_btn = gr.Button(

                "🚀 Generate Image",

                variant="primary",

                size="lg",
            )


            # ------------------------------------------------
            # STATUS
            # ------------------------------------------------

            status = gr.Textbox(

                label="Status",

                lines=7,

                interactive=False,
            )


        # ====================================================
        # RIGHT COLUMN
        # ====================================================

        with gr.Column(
            scale=1
        ):


            output_image = gr.Image(

                label="Generated Image",

                type="pil",
            )


    # ========================================================
    # EXAMPLES
    # ========================================================

    gr.Markdown(
        "### Example Prompts"
    )


    gr.Examples(

        examples=[

            [
                (
                    "Photorealistic portrait of an Indian "
                    "woman wearing an elegant red saree, "
                    "natural skin texture, realistic eyes, "
                    "soft golden sunlight, professional "
                    "fashion photography"
                )
            ],

            [
                (
                    "A futuristic Mumbai city in 2050 "
                    "during heavy rain at night, realistic "
                    "skyscrapers, wet streets reflecting "
                    "neon lights, cinematic photography, "
                    "realistic atmosphere"
                )
            ],

            [
                (
                    "A peaceful Himalayan mountain valley "
                    "at sunrise, natural atmospheric haze, "
                    "distant snow-covered peaks, realistic "
                    "landscape photography, golden morning "
                    "light"
                )
            ],

            [
                (
                    "A premium wireless headphone product "
                    "on a minimalist wooden desk, realistic "
                    "materials, soft studio lighting, "
                    "commercial product photography, "
                    "sharp details"
                )
            ],

        ],

        inputs=prompt,
    )


    # ========================================================
    # BUTTON EVENT
    # ========================================================

    generate_btn.click(

        fn=generate_image,

        inputs=[

            prompt,

            steps,

            width,

            height,

            seed,
        ],

        outputs=[

            output_image,

            status,
        ],
    )


    # ========================================================
    # ENTER KEY
    # ========================================================

    prompt.submit(

        fn=generate_image,

        inputs=[

            prompt,

            steps,

            width,

            height,

            seed,
        ],

        outputs=[

            output_image,

            status,
        ],
    )


# ================================================================
# START GRADIO
# ================================================================

print()
print("=" * 70)
print("✅ MODEL + UI READY")
print("=" * 70)

print()
print(
    "Starting Gradio..."
)

print(
    "A public URL will appear below."
)

print(
    "Keep this Kaggle session running."
)

print()


# ------------------------------------------------------------
# Queue requests.
#
# The Python lock also ensures one generation at a time.
# ------------------------------------------------------------

demo.queue()


# ------------------------------------------------------------
# Launch.
#
# share=True gives the public Gradio URL.
# ------------------------------------------------------------

demo.launch(

    share=True,

    server_name="0.0.0.0",

    server_port=7860,

    show_error=True,
)
'''


# ================================================================
# WRITE app.py
# ================================================================

with open(
    APP_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(app_code)


print(
    "✅ Created:",
    APP_FILE
)


# ================================================================
# 6. LAUNCH app.py
# ================================================================

print()
print("=" * 70)
print("STEP 6/6 — LAUNCHING QWEN IMAGE SERVER")
print("=" * 70)

print()
print(
    "⏳ Qwen-Image-2.1 will load now."
)

print(
    "⏳ After the model loads, Gradio will display the public URL."
)

print()
print(
    "⚠️ DO NOT STOP THE CELL while you are using the web UI."
)

print()


# ------------------------------------------------
# Start child process.
# ------------------------------------------------

process = subprocess.Popen(

    [
        sys.executable,

        "-u",

        APP_FILE,
    ],

    stdout=subprocess.PIPE,

    stderr=subprocess.STDOUT,

    text=True,

    bufsize=1,
)


# ------------------------------------------------
# Stream output live.
# ------------------------------------------------

try:

    for line in process.stdout:

        print(
            line,
            end=""
        )


except KeyboardInterrupt:

    print()
    print(
        "🛑 Stopping Qwen server..."
    )


    try:

        process.terminate()

        process.wait(
            timeout=10
        )

    except Exception:

        try:
            process.kill()
        except Exception:
            pass


    print(
        "✅ Server stopped."
    )